# Exp-4 — Cross-encoder rerank with bge-reranker-v2-m3

**Why this exp.** Exp-3 (Qwen3-32B + HyDE + Enum) reached `hyde+enum stat_recall@500 = 0.376` and `@1000 = 0.456`. The gold *is* in the candidate list — it just sits too deep. A reranker's job is to surface it into the top-K that we can actually submit.

**Setup.**
- Candidate set: hyde+enum top-1000 per query (reconstructed from `query_vecs_A3.npz` + `laws_bgem3.npy`).
- Reranker: `BAAI/bge-reranker-v2-m3` (multilingual, fp16).
- Two query forms tested: translated German (primary — matches doc language) and raw English (ablation).
- Output: statute recall@{10, 20, 30, 50, 100, 200} after rerank, compared to pre-rerank baseline at the same K.

**What we're watching.**
1. Does rerank **move gold up**? If `recall@50` after rerank > `recall@500` before rerank (0.376), the reranker is earning its keep.
2. Does DE query beat EN? Reranker is multilingual but not zero-shot magic — translated DE should win if translation noise isn't too bad.
3. Per-query: which of val_008 / val_010 / val_003 stay broken? Those signal the case-retrieval gap, not a rerank failure.

**Gate.** `stat_recall@50` after rerank ≥ 0.35 is a useful floor for a ≈25-citation submission. If we don't reach it, the gold density in the top-1000 isn't high enough — widen the candidate set (top-2000) or add case retrieval before reranking.

In [12]:
# --- Cell 1. Install deps ---
!pip install -qU numpy==1.26.4
!pip install -qU FlagEmbedding pandas numpy
# Restart runtime after installing numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gradio 5.50

In [2]:
# --- Cell 2. Mount Drive & paths ---
from google.colab import drive
drive.mount('/content/drive')

import json, pickle, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/swiss_law/data')
ART  = ROOT / 'artifacts'
for f in ['laws_bgem3.npy', 'query_vecs_A3.npz', 'exp_A3_expansions.json']:
    assert (ART / f).exists(), f'missing {f} — run Exp-1/3 first'

assert torch.cuda.is_available(), 'need GPU (T4/L4/A100 all work for reranker)'
print('GPU:', torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
# --- Cell 3. Load data + rebuild doc text ---
val  = pd.read_csv(ROOT / 'val.csv')
laws = pd.read_csv(ROOT / 'laws_de.csv')
with open(ROOT / 'val_translated_de.pkl', 'rb') as f:
    val_de = pickle.load(f)
with open(ART / 'exp_A3_expansions.json', 'r', encoding='utf-8') as f:
    exp = json.load(f)

cits = laws['citation'].tolist()
# reranker sees the same text that the retriever indexed: citation | title | text
docs = (laws['citation'].fillna('') + ' | ' +
        laws['title'].fillna('')     + ' | ' +
        laws['text'].fillna('')).tolist()

doc_emb = np.load(ART / 'laws_bgem3.npy').astype(np.float32)
qv = np.load(ART / 'query_vecs_A3.npz')
q_en, q_de, q_hyde, q_enum = qv['q_en'], qv['q_de'], qv['q_hyde'], qv['q_enum']
query_ids = [str(x) for x in qv['query_ids']]
assert query_ids == list(val['query_id']), 'order mismatch'
print('corpus:', doc_emb.shape, '| val:', len(val))

corpus: (175933, 1024) | val: 10


In [4]:
# --- Cell 4. Rebuild hyde+enum top-1000 candidates (RRF, same as Exp-3) ---
def rank_all(q):
    sims = q @ doc_emb.T
    return np.argsort(-sims, axis=1)

rank_hyde = rank_all(q_hyde)
rank_enum = rank_all(q_enum)

def rrf(rank_lists, k_rrf=60, topk=1000):
    N_q, N_doc = rank_lists[0].shape
    scores = np.zeros((N_q, N_doc), dtype=np.float32)
    for rl in rank_lists:
        pos = np.empty_like(rl)
        rows = np.arange(N_q)[:, None]
        pos[rows, rl] = np.arange(N_doc)[None, :]
        scores += 1.0 / (k_rrf + pos.astype(np.float32))
    idx = np.argpartition(-scores, topk - 1, axis=1)[:, :topk]
    rows = np.arange(N_q)[:, None]
    order = np.argsort(-scores[rows, idx], axis=1)
    return idx[rows, order]

cands = rrf([rank_hyde, rank_enum], topk=1000)  # (10, 1000)
print('candidates:', cands.shape)

candidates: (10, 1000)


In [1]:
# --- Cell 5. Load reranker ---
!pip install -q "transformers<4.45.0"
from FlagEmbedding import FlagReranker
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [5]:
# --- Cell 6. Rerank each query × 1000 candidates, two query forms ---
def rerank_for_queries(query_texts, label):
    t0 = time.time()
    out_scores = np.zeros((len(query_texts), 1000), dtype=np.float32)
    for i, q in enumerate(query_texts):
        pairs = [[q, docs[cands[i, j]]] for j in range(1000)]
        # batched scoring internally; bge-reranker-v2-m3 handles up to 8192 tok
        s = reranker.compute_score(pairs, batch_size=32, max_length=1024, normalize=True)
        out_scores[i] = np.asarray(s, dtype=np.float32)
        print(f'  {label} [{i+1}/{len(query_texts)}] {val.iloc[i]["query_id"]}: {time.time()-t0:.0f}s elapsed')
    print(f'{label} total: {time.time()-t0:.0f}s')
    return out_scores

en_queries = [exp[qid]['query_en'] for qid in query_ids]
de_queries = [val_de[qid] for qid in query_ids]

scores_de = rerank_for_queries(de_queries, 'DE')
scores_en = rerank_for_queries(en_queries, 'EN')

np.savez(ART / 'rerank_scores_A4.npz',
         cands=cands, scores_de=scores_de, scores_en=scores_en,
         query_ids=np.array(query_ids))

Compute Scores: 100%|██████████| 32/32 [00:02<00:00, 15.16it/s]


  DE [1/10] val_001: 3s elapsed


Compute Scores: 100%|██████████| 32/32 [00:03<00:00,  9.22it/s]


  DE [2/10] val_002: 7s elapsed


Compute Scores: 100%|██████████| 32/32 [00:03<00:00,  8.58it/s]


...

  EN [8/10] val_008: 24s elapsed


Compute Scores: 100%|██████████| 32/32 [00:02<00:00, 14.58it/s]


  EN [9/10] val_009: 26s elapsed


Compute Scores: 100%|██████████| 32/32 [00:03<00:00, 10.13it/s]

  EN [10/10] val_010: 30s elapsed
EN total: 30s


In [6]:
# --- Cell 7. Evaluate: before vs after rerank ---
def parse(s): return [c.strip() for c in str(s).split(';') if c.strip()]
def is_stat(c):
    return not (c.startswith('BGE ') or re.match(r'\d[A-Z]_', c) or re.match(r'[A-Z]\d[A-Z]_', c))

def eval_at_k(order_per_q, cands, label, ks=(10, 20, 30, 50, 100, 200, 500)):
    per_q = []
    for i, row in enumerate(val.itertuples()):
        gold = {c for c in parse(row.gold_citations) if is_stat(c)}
        ranked_docs = cands[i, order_per_q[i]]  # doc indices in new order
        ranked_cits = [cits[j] for j in ranked_docs]
        entry = {'query_id': row.query_id, 'n_gold': len(gold)}
        for k in ks:
            entry[f'hit@{k}'] = len(gold & set(ranked_cits[:k]))
        per_q.append(entry)
    agg = {f'stat_recall@{k}': sum(p[f'hit@{k}'] for p in per_q)
                              / max(1, sum(p['n_gold'] for p in per_q)) for k in ks}
    print(f'=== {label} ===')
    for k, v in agg.items():
        print(f'  {k} = {v:.3f}')
    return {'agg': agg, 'per_query': per_q}

# Baseline: RRF order (no rerank) — order_per_q = identity [0..999]
baseline_order = np.tile(np.arange(1000), (len(val), 1))

# Reranked: sort candidates by descending score
order_de = np.argsort(-scores_de, axis=1)
order_en = np.argsort(-scores_en, axis=1)

report = {
    'baseline_rrf':  eval_at_k(baseline_order, cands, 'baseline (hyde+enum RRF, no rerank)'),
    'rerank_de':     eval_at_k(order_de,       cands, 'rerank with DE query'),
    'rerank_en':     eval_at_k(order_en,       cands, 'rerank with EN query'),
}

=== baseline (hyde+enum RRF, no rerank) ===
  stat_recall@10 = 0.047
  stat_recall@20 = 0.101
  stat_recall@30 = 0.121
  stat_recall@50 = 0.161
  stat_recall@100 = 0.255
  stat_recall@200 = 0.315
  stat_recall@500 = 0.376
=== rerank with DE query ===
  stat_recall@10 = 0.047
  stat_recall@20 = 0.087
  stat_recall@30 = 0.101
  stat_recall@50 = 0.114
  stat_recall@100 = 0.148
  stat_recall@200 = 0.215
  stat_recall@500 = 0.342
=== rerank with EN query ===
  stat_recall@10 = 0.027
  stat_recall@20 = 0.040
  stat_recall@30 = 0.060
  stat_recall@50 = 0.067
  stat_recall@100 = 0.094
  stat_recall@200 = 0.148
  stat_recall@500 = 0.336


In [7]:
# --- Cell 8. Per-query breakdown: which queries get fixed, which stay broken ---
print(f'{"qid":<10} {"gold":>4} | {"base@50":>8} {"de@50":>6} {"en@50":>6} | {"de@30":>6} {"de@20":>6} {"de@10":>6}')
print('-' * 70)
for i, row in enumerate(val.itertuples()):
    qid = row.query_id
    b = report['baseline_rrf']['per_query'][i]
    d = report['rerank_de']['per_query'][i]
    e = report['rerank_en']['per_query'][i]
    print(f'{qid:<10} {b["n_gold"]:>4} | '
          f'{b["hit@50"]:>8} {d["hit@50"]:>6} {e["hit@50"]:>6} | '
          f'{d["hit@30"]:>6} {d["hit@20"]:>6} {d["hit@10"]:>6}')

qid        gold |  base@50  de@50  en@50 |  de@30  de@20  de@10
----------------------------------------------------------------------
val_001      19 |        6      5      4 |      5      5      4
val_002      20 |        3      1      0 |      0      0      0
val_003      24 |        3      2      0 |      2      1      0
val_004       9 |        2      3      3 |      3      3      1
val_005       6 |        2      1      1 |      1      1      0
val_006      11 |        4      0      0 |      0      0      0
val_007      15 |        1      1      0 |      1      0      0
val_008      20 |        0      1      1 |      1      1      1
val_009      11 |        1      3      1 |      2      2      1
val_010      14 |        2      0      0 |      0      0      0


In [8]:
# --- Cell 9. Save report + verdict ---
report['meta'] = {
    'reranker': 'BAAI/bge-reranker-v2-m3',
    'candidate_source': 'Exp-3 hyde+enum top-1000 (Qwen3-32B expansions)',
    'n_queries': len(val),
    'baselines': {
        'exp1_en_recall@500': 0.215,
        'exp2_7b_hyde+enum_recall@500': 0.295,
        'exp3_32b_hyde+enum_recall@500': 0.376,
        'exp3_32b_hyde+enum_recall@1000': 0.456,
    },
    'useful_floor': 'stat_recall@50 >= 0.35 (sub-25-citation submission)'
}
with open(ART / 'exp_A4_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print('=' * 74)
print(f'{"variant":<40} {"@10":>6} {"@20":>6} {"@30":>6} {"@50":>6} {"@100":>6} {"@200":>6}')
print('-' * 74)
for k in ['baseline_rrf', 'rerank_de', 'rerank_en']:
    a = report[k]['agg']
    print(f"{k:<40} {a['stat_recall@10']:>6.3f} {a['stat_recall@20']:>6.3f} "
          f"{a['stat_recall@30']:>6.3f} {a['stat_recall@50']:>6.3f} "
          f"{a['stat_recall@100']:>6.3f} {a['stat_recall@200']:>6.3f}")

best = max(report['rerank_de']['agg']['stat_recall@50'],
           report['rerank_en']['agg']['stat_recall@50'])
print(f'\nbest rerank stat_recall@50 = {best:.3f}')
print(f'(pre-rerank @50 = {report["baseline_rrf"]["agg"]["stat_recall@50"]:.3f})')
print(f'floor 0.35 -> {"PASS" if best >= 0.35 else "STILL BELOW"}')

variant                                     @10    @20    @30    @50   @100   @200
--------------------------------------------------------------------------
baseline_rrf                              0.047  0.101  0.121  0.161  0.255  0.315
rerank_de                                 0.047  0.087  0.101  0.114  0.148  0.215
rerank_en                                 0.027  0.040  0.060  0.067  0.094  0.148

best rerank stat_recall@50 = 0.114
(pre-rerank @50 = 0.161)
floor 0.35 -> STILL BELOW
